# Step 6 — Plot and export as a STAC Catalog

Draws a quicklook of the biomass map and packages the results as a self-contained
STAC Catalog with one Item, the shape expected at the end of an OGC Application Package
(BP 20-089). The Item links to the source Sentinel-2 scene (`derived_from`).

| | |
|---|---|
| Six-phase position | Export / aggregation |
| W1 Algae Bloom counterpart | `plot-image` (colourised map + PNG) |
| Output | `stac_catalog`: directory `outputs/` holding `catalog.json` and the Item |

Same Catalog layout as the single-notebook version (`mangrove-biomass-analysis`, one Item
`mangrove-analysis-<timestamp>`), with two more assets: `biomass.tif` and the quicklook.

In [ ]:
import json
import os
import shutil
from datetime import datetime, timezone

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pystac
import rasterio

In [ ]:
# CWL type annotations (removed by ipython2cwl in the generated tool)
from typing import List, Optional

from ipython2cwl.iotypes import (
    CWLDirectoryPathOutput,
    CWLFilePathInput,
    CWLFilePathOutput,
    CWLFloatInput,
    CWLIntInput,
    CWLMetadata,
    CWLNamespaces,
    CWLRequirement,
    CWLStringInput,
)

In [ ]:
cwl_requirements: CWLRequirement = {
    "ResourceRequirement": {"coresMin": 1, "ramMin": 512},
}

In [ ]:
cwl_metadata: CWLMetadata = {
    "s:softwareVersion": "0.1.0",
    "s:keywords": ["ospd", "mangrove", "stac", "export"],
    "s:author": [{"class": "s:Person", "s:name": "Cameron Sajedi"}],
    "s:contributor": [
        {"class": "s:Person", "s:name": "Gérald Fenoy", "s:affiliation": "GeoLabs"}
    ],
    "s:codeRepository": "https://github.com/starling-foundries/KindGrove",
    "s:license": "https://spdx.org/licenses/CC-BY-NC-SA-4.0",
    "s:description": "Plot the biomass map and export the results as a STAC Catalog",
}

In [ ]:
cwl_namespaces: CWLNamespaces = {
    "s": "https://schema.org/",
}

## Inputs

In [ ]:
mangrove_mask_file: CWLFilePathInput = "mangrove_mask.tif"
biomass_file: CWLFilePathInput = "biomass.tif"
biomass_summary_file: CWLFilePathInput = "biomass_summary.csv"
carbon_summary_file: CWLFilePathInput = "carbon_summary.csv"
analysis_summary_file: CWLFilePathInput = "analysis_summary.json"
stac_item: CWLFilePathInput = "scene_item.json"
west: CWLFloatInput = 95.15
south: CWLFloatInput = 15.9
east: CWLFloatInput = 95.35
north: CWLFloatInput = 16.1

## Quicklook (plot)

In [ ]:
with rasterio.open(biomass_file) as src:
    biomass = src.read(1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

fig, ax = plt.subplots(figsize=(8, 8))
image = ax.imshow(np.ma.masked_invalid(biomass), cmap="YlGn", extent=extent, vmin=0)
fig.colorbar(image, ax=ax, shrink=0.7, label="Above-ground biomass (Mg/ha)")
ax.set_title("Mangrove above-ground biomass")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
quicklook_file = "biomass_quicklook.png"
fig.savefig(quicklook_file, dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {quicklook_file}")

## STAC Catalog

In [ ]:
with open(analysis_summary_file) as f:
    summary = json.load(f)
with open(stac_item) as f:
    scene = json.load(f)

stac_catalog: CWLDirectoryPathOutput = "outputs"
output_dir = stac_catalog
bbox = [west, south, east, north]
now = datetime.now(timezone.utc)
catalog = pystac.Catalog(
    id="mangrove-biomass-analysis",
    description="Mangrove biomass and carbon analysis from Sentinel-2 imagery",
    title="Mangrove Biomass Analysis",
)
item_id = f"mangrove-analysis-{now:%Y%m%d-%H%M%S}"
item = pystac.Item(
    id=item_id,
    geometry={
        "type": "Polygon",
        "coordinates": [[
            [west, south], [east, south], [east, north], [west, north], [west, south]
        ]],
    },
    bbox=bbox,
    datetime=now,
    properties={
        "scene_id": summary["scene_id"],
        "scene_datetime": summary["scene_datetime"],
        "cloud_cover": summary["cloud_cover"],
        "mangrove_area_ha": summary["mangrove_area_ha"],
        "biomass_tons": summary["biomass_tons"],
        "carbon_tons": summary["carbon_tons"],
    },
)
scene_self = [link["href"] for link in scene.get("links", []) if link.get("rel") == "self"]
if scene_self:
    item.add_link(pystac.Link("derived_from", scene_self[0], media_type="application/geo+json", title=scene["id"]))

item_dir = os.path.join(output_dir, item_id)
os.makedirs(item_dir, exist_ok=True)
assets = [
    (mangrove_mask_file, "mangrove_mask.tif", "image/tiff; application=geotiff", "Mangrove detection mask", ["data"]),
    (biomass_file, "biomass.tif", "image/tiff; application=geotiff", "Above-ground biomass (Mg/ha)", ["data"]),
    (biomass_summary_file, "biomass_summary.csv", "text/csv", "Biomass analysis summary", ["metadata"]),
    (carbon_summary_file, "carbon_summary.csv", "text/csv", "Carbon storage analysis", ["metadata"]),
    (quicklook_file, "biomass_quicklook.png", "image/png", "Biomass quicklook", ["overview"]),
]
for source, filename, media_type, title, roles in assets:
    shutil.copy(source, os.path.join(item_dir, filename))
    item.add_asset(
        filename.replace(".", "_"),
        pystac.Asset(href=filename, media_type=media_type, title=title, roles=roles),
    )

catalog.add_item(item)
catalog.normalize_and_save(root_href=output_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
print(f"STAC catalog saved to {os.path.join(output_dir, 'catalog.json')}")